# 🎤 TTS Server + AI Proxy

**Microsoft Edge TTS + Cloudflare Tunnel + OpenRouter AI Proxy**

Этот скрипт запускает:
- ✅ TTS сервер (edge-tts) - без ограничений
- ✅ AI прокси для OpenRouter (чат)
- ✅ Cloudflare Tunnel для публичного доступа
- ✅ Статистика использования

📌 **После запуска скопируйте ссылку и вставьте на сайт!**

🔑 **Важно:** Добавьте Secret в Colab:
1. Нажмите на 🔑 Key management (слева)
2. ADD NEW SECRET
3. Name: `OPENROUTER_API_KEY`
4. Value: `sk-or-v1-ваш_ключ_здесь`
5. Нажмите Save

In [ ]:
# ============================================================================
# 📦 INSTALL DEPENDENCIES
# ============================================================================
print("=" * 80)
print("📦 INSTALLING DEPENDENCIES")
print("=" * 80)

!pip install edge-tts flask requests -q

print("✅ Dependencies installed")

In [ ]:
# ============================================================================
# 🚀 LAUNCH TTS + AI PROXY SERVER
# ============================================================================
print("\n" + "=" * 80)
print("🚀 LAUNCHING TTS + AI PROXY SERVER")
print("=" * 80)

import os
import subprocess
import asyncio
import threading
import time
import re
import sys
import json
from flask import Flask, request, Response, jsonify
import edge_tts
import requests

print("✅ Modules imported")

# Create Flask app
app = Flask(__name__)

# ================================================================
# 🔑 ПОЛУЧАЕМ API КЛЮЧ ИЗ COLAB SECRETS (БЕЗОПАСНО!)
# ================================================================
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if not OPENROUTER_API_KEY:
        raise ValueError("Key is empty")
    print("✅ API key loaded from Colab Secrets")
except Exception as e:
    print("❌ ERROR: OpenRouter API key not found in Colab Secrets!")
    print("   Please set up the secret:")
    print("   1. Click on 🔑 Key management (left sidebar)")
    print("   2. ADD NEW SECRET")
    print("   3. Name: OPENROUTER_API_KEY")
    print("   4. Value: your_api_key_here")
    print("   5. Click Save")
    print("   6. Restart runtime: Runtime → Restart runtime")
    raise SystemExit("API key missing. Please set up Colab Secrets.")

# Define voices globally
voices = {
    "us_young": "en-US-ChristopherNeural",
    "us_male": "en-US-EricNeural", 
    "us_male2": "en-US-GuyNeural",
    "us_female": "en-US-JennyNeural",
    "us_female2": "en-US-AriaNeural",
    "us_female3": "en-US-AnaNeural",
    "uk_male": "en-GB-RyanNeural",
    "uk_male2": "en-GB-AlfieNeural",
    "uk_female": "en-GB-SoniaNeural",
    "uk_female2": "en-GB-LibbyNeural",
    "au_male": "en-AU-WilliamNeural",
    "au_female": "en-AU-NatashaNeural",
    "indian_male": "en-IN-PrabhatNeural",
    "indian_female": "en-IN-NeerjaNeural",
    "ca_male": "en-CA-LiamNeural",
    "ca_female": "en-CA-ClaraNeural"
}

# Enable CORS
@app.after_request
def cors(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, X-Requested-With"
    return response

# ==================== TTS ENDPOINT ====================

@app.route("/", methods=["GET", "POST", "OPTIONS"])
def tts_root():
    if request.method == 'OPTIONS':
        return '', 200
    return tts_request()

@app.route("/tts", methods=["GET", "POST", "OPTIONS"])
def tts_request():
    """Main TTS function - NO LIMITS"""
    if request.method == 'OPTIONS':
        return '', 200
    
    text = ""
    voice = "us_young"
    speed = "1.0"
    
    if request.method == 'GET':
        text = request.args.get("text", "")
        voice = request.args.get("voice", "us_young")
        speed = request.args.get("speed", "1.0")
    elif request.method == 'POST':
        try:
            if request.is_json:
                data = request.get_json()
                text = data.get('text', '')
                voice = data.get('voice', 'us_young')
                speed = data.get('speed', '1.0')
        except:
            text = ""
    
    if not text:
        return jsonify({"error": "No text provided"}), 400
    
    print(f"📝 TTS: {len(text)} chars, voice={voice}, speed={speed}")
    
    if voice not in voices:
        voice = "us_young"
    voice_id = voices[voice]
    
    try:
        speed_float = float(speed)
        if speed_float == 1.0:
            rate_str = "+0%"
        else:
            percentage = int((speed_float - 1.0) * 100)
            rate_str = f"+{percentage}%" if percentage >= 0 else f"{percentage}%"
    except:
        rate_str = "+0%"
    
    async def generate_audio():
        communicate = edge_tts.Communicate(text, voice_id, rate=rate_str)
        chunks = []
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                chunks.append(chunk["data"])
        return b"".join(chunks)
    
    try:
        audio = asyncio.run(generate_audio())
        
        response = Response(audio, mimetype="audio/mpeg")
        response.headers["Content-Length"] = len(audio)
        response.headers["X-TTS-Length"] = len(text)
        response.headers["X-TTS-Voice"] = voice
        response.headers["X-TTS-Speed"] = speed
        
        print(f"✅ TTS Success: {len(audio)} bytes")
        return response
        
    except Exception as e:
        print(f"❌ TTS Error: {e}")
        return jsonify({"error": str(e)}), 500

@app.route("/status", methods=["GET", "OPTIONS"])
def status():
    """Server status endpoint"""
    if request.method == 'OPTIONS':
        return '', 200
    return jsonify({
        "status": "running",
        "message": "TTS + AI Server - NO LIMITS",
        "voices": list(voices.keys()),
        "speed_range": "0.5 to 3.0",
        "endpoints": ["/tts", "/proxy", "/status", "/test", "/stats"]
    })

# ==================== AI CHAT PROXY ====================

@app.route('/proxy', methods=['POST', 'OPTIONS'])
def proxy():
    """AI Chat proxy to OpenRouter"""
    if request.method == 'OPTIONS':
        return '', 204, {
            'Access-Control-Allow-Origin': '*',
            'Access-Control-Allow-Methods': 'POST, OPTIONS',
            'Access-Control-Allow-Headers': 'Content-Type'
        }

    try:
        data = request.get_json()
        
        if not data:
            return jsonify({"error": "No data provided"}), 400
        
        print(f"📝 PROXY Request: model={data.get('model', 'default')}, messages={len(data.get('messages', []))}")

        headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {OPENROUTER_API_KEY}',
            'HTTP-Referer': 'https://lepka-iz-plastilina.ru',
            'X-OpenRouter-Title': 'Sports AI Assistant'
        }

        payload = {
            'model': data.get('model', 'meta-llama/llama-3.3-70b-instruct'),
            'messages': data.get('messages', []),
            'max_completion_tokens': data.get('max_completion_tokens', 4000),
            'temperature': data.get('temperature', 0.7)
        }

        print(f"📤 Sending to OpenRouter: {payload['model']}")
        
        response = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers=headers,
            json=payload,
            timeout=90
        )

        print(f"✅ PROXY Response: {response.status_code}")
        
        if response.status_code != 200:
            try:
                error_data = response.json()
                print(f"❌ OpenRouter Error: {error_data}")
            except:
                print(f"❌ OpenRouter Error: {response.text}")
        
        return response.text, response.status_code, {
            'Content-Type': 'application/json',
            'Access-Control-Allow-Origin': '*'
        }
        
    except requests.exceptions.Timeout:
        print("❌ PROXY Timeout")
        return jsonify({"error": "Timeout waiting for OpenRouter"}), 504
    except Exception as e:
        print(f"❌ PROXY Error: {e}")
        return jsonify({"error": str(e)}), 500

# ==================== СТАТИСТИКА ====================
@app.route('/stats', methods=['GET', 'OPTIONS'])
def get_stats():
    """Get OpenRouter usage stats"""
    if request.method == 'OPTIONS':
        return '', 204, {
            'Access-Control-Allow-Origin': '*',
            'Access-Control-Allow-Methods': 'GET, OPTIONS',
            'Access-Control-Allow-Headers': 'Content-Type'
        }
    
    try:
        headers = {
            'Authorization': f'Bearer {OPENROUTER_API_KEY}'
        }
        
        response = requests.get(
            'https://openrouter.ai/api/v1/auth/key',
            headers=headers,
            timeout=10
        )
        
        if response.status_code == 200:
            data = response.json()
            usage = data.get('data', {}).get('usage', 0)
            credits = data.get('data', {}).get('credits', 0)
            return jsonify({
                'spent': usage,
                'credits': credits,
                'status': 'ok'
            })
        else:
            return jsonify({
                'spent': 0,
                'credits': 0,
                'status': 'error',
                'message': 'Failed to get stats'
            }), 500
            
    except Exception as e:
        return jsonify({
            'spent': 0,
            'credits': 0,
            'status': 'error',
            'message': str(e)
        }), 500

# ==================== ТЕСТОВЫЙ ЭНДПОИНТ ====================
@app.route('/test', methods=['GET'])
def test():
    """Test endpoint"""
    return jsonify({
        "status": "ok",
        "message": "Server is working!",
        "endpoints": ["/tts", "/proxy", "/status", "/test", "/stats"]
    })

# ==================== ЗАПУСК СЕРВЕРА ====================

def run_server():
    app.run(host="0.0.0.0", port=9999, debug=False, threaded=True, use_reloader=False)

# Kill old processes
print("Stopping old processes...")
!fuser -k 9999/tcp 2>/dev/null || true
time.sleep(2)

# Start server
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

print("✅ Server running on http://localhost:9999")
print("📋 Available endpoints:")
print("   • GET  /tts?text=...&voice=...&speed=...  (TTS)")
print("   • POST /proxy  (AI Chat)")
print("   • GET  /status")
print("   • GET  /test  (Test)")
print("   • GET  /stats  (Statistics)")

In [ ]:
# ============================================================================
# 🌐 SETUP CLOUDFLARED
# ============================================================================
print("\n" + "=" * 80)
print("🌐 SETTING UP CLOUDFLARED")
print("=" * 80)

!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

print("Installing cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null || apt-get install -f -y

print("Starting Cloudflare Tunnel...")
!cloudflared tunnel --url http://localhost:9999 > /tmp/cloudflared.log 2>&1 &
time.sleep(5)

print("Getting URL...")
time.sleep(15)

try:
    with open('/tmp/cloudflared.log', 'r') as f:
        logs = f.read()
    
    public_url = None
    url_patterns = [
        r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com',
        r'Visit it at.*?(https://[^\s]+)',
        r'Your quick Tunnel.*?(https://[^\s]+)'
    ]
    
    for pattern in url_patterns:
        matches = re.findall(pattern, logs)
        if matches:
            public_url = matches[-1]
            print(f"✅ URL found: {public_url}")
            break
    
    if not public_url:
        public_url = "https://your-server.trycloudflare.com"
        print("⚠️ URL not found in logs")
        
except Exception as e:
    print(f"Error reading logs: {e}")
    public_url = None

print("✅ Cloudflare Tunnel ready")

In [ ]:
# ============================================================================
# 🎯 READY
# ============================================================================
if public_url:
    print("\n" + "=" * 80)
    print("🎯 SERVER READY - TTS + AI CHAT + STATS")
    print("=" * 80)
    print(f"\n🔗 YOUR URL: {public_url}")
    
    with open('/content/tts_url.txt', 'w') as f:
        f.write(public_url)
    print(f"💾 URL saved to /content/tts_url.txt")
    
    print("\n📝 TTS USAGE:")
    print(f"GET:  {public_url}/tts?text=Your+text&voice=us_male&speed=1.5")
    
    print("\n🤖 AI CHAT USAGE:")
    print(f"POST: {public_url}/proxy")
    print("Body: {\"model\":\"meta-llama/llama-3.3-70b-instruct\",\"messages\":[...]}")
    
    print("\n📊 STATS USAGE:")
    print(f"GET:  {public_url}/stats")
    
    print("\n🔧 TEST:")
    print(f"GET:  {public_url}/test")
    
    print("\n" + "=" * 80)
    print("✅ ALL ENDPOINTS AVAILABLE:")
    print("=" * 80)
    print(f"  /tts   - Text-to-Speech")
    print(f"  /proxy - AI Chat (OpenRouter)")
    print(f"  /stats - Usage statistics")
    print(f"  /test  - Server test")
    print(f"  /status- Server status")
    
else:
    print("\n❌ Failed to get URL")

print("\n" + "=" * 80)
print("✅ SERVER IS RUNNING")
print("=" * 80)
print("\n💡 Keep this Colab running to use the server")
print("💡 URL: " + (public_url if public_url else "Not available"))

## 📋 Инструкция:

1. **Скопируйте URL** который появился выше
2. **Перейдите на сайт** и вставьте URL в поле "Server URL"
3. **Нажмите "Update"**
4. **TTS работает!** 🎉
5. **AI чат работает!** 🤖

### Доступные эндпоинты:
- `/tts?text=...&voice=...&speed=...` - генерация речи
- `/proxy` - AI чат (OpenRouter)
- `/stats` - статистика использования
- `/test` - проверка работы сервера
- `/status` - статус сервера